In [1]:
from sklearn.datasets import load_wine
import torch
from torch.utils.data import DataLoader
import torch.nn as nn
from torch.utils.data import Dataset
import math
import torch.nn.functional as F
import torch.optim as optim


D:\ANACONDA\Lib\site-packages\torch\cuda\__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


In [89]:
wine = load_wine()

X = wine.data
y = wine.target

In [91]:
print(X[0,:])

[1.423e+01 1.710e+00 2.430e+00 1.560e+01 1.270e+02 2.800e+00 3.060e+00
 2.800e-01 2.290e+00 5.640e+00 1.040e+00 3.920e+00 1.065e+03]


In [93]:
from torch.utils.data import random_split

In [95]:
import pandas as pd
print(X.shape)
print(y.shape)
print(y)
#X=pd.DataFrame(X)
#print(X.head())

(178, 13)
(178,)
[0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2
 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2]


In [97]:
#y=pd.DataFrame(y)
#print(y.head())

In [99]:
#print(y.isnull().sum ==True)
#print(X.isnull().sum==True)

In [101]:
#print(X.head())

In [103]:
from sklearn.preprocessing import StandardScaler

In [107]:
scaler=StandardScaler()
X=scaler.fit_transform(X)

In [109]:
X=torch.tensor(X,dtype=torch.float32)
y=torch.tensor(y,dtype=torch.long)

In [129]:
class mymodel(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1=nn.Linear(13,32)
        self.fc2=nn.Linear(32,16)
        self.fc3=nn.Linear(16,3)

    def forward(self,x):
        x=F.relu(self.fc1(x))
        x=F.relu(self.fc2(x))
        x=self.fc3(x)
        return x

In [131]:
class Winedataset(Dataset):
    def __init__(self,x,y):
       # super().__init__()
        self.x=x
        self.y=y
        self.n_samples=len(self.x)

    def __len__(self):
        return self.n_samples

    def __getitem__(self,index):
        return self.x[index], self.y[index]

In [133]:
model=mymodel()
criterion=nn.CrossEntropyLoss()
optimiser=optim.Adam(model.parameters(),lr=0.01)


In [135]:
dataset=Winedataset(X,y)
train_size=int(0.8*len(dataset))
test_size=len(dataset)-train_size
train_dataset,test_dataset=random_split(dataset, [train_size,test_size])


In [139]:
train_loader=DataLoader(train_dataset,batch_size=16,shuffle=True)
for epoch in range(75):
    epoch_loss=0
    for x_train,y_train in train_loader:
        optimiser.zero_grad()
        output=model(x_train)
        loss=criterion(output,y_train)
        loss.backward()
        optimiser.step()
        epoch_loss=epoch_loss+loss.item()
        
    print("Loss for Epoch ",epoch,":",epoch_loss)


Loss for Epoch  0 : 0.0003905715166183654
Loss for Epoch  1 : 0.0003684781299853057
Loss for Epoch  2 : 0.0003718090060829127
Loss for Epoch  3 : 0.0003558333714863693
Loss for Epoch  4 : 0.00035644312902149977
Loss for Epoch  5 : 0.00034425875628585345
Loss for Epoch  6 : 0.00033085536597354803
Loss for Epoch  7 : 0.00032802711393742356
Loss for Epoch  8 : 0.0003218806368749938
Loss for Epoch  9 : 0.0003160748351547227
Loss for Epoch  10 : 0.0003047222508030245
Loss for Epoch  11 : 0.00030072676406689425
Loss for Epoch  12 : 0.00029303191104190773
Loss for Epoch  13 : 0.00028625156619455083
Loss for Epoch  14 : 0.0002804250343615422
Loss for Epoch  15 : 0.0002725946051214123
Loss for Epoch  16 : 0.00026643138289728086
Loss for Epoch  17 : 0.0002683648694983276
Loss for Epoch  18 : 0.0002584747990113101
Loss for Epoch  19 : 0.0002520134958103881
Loss for Epoch  20 : 0.00024651953754073475
Loss for Epoch  21 : 0.00024224031449193717
Loss for Epoch  22 : 0.00023640440974759258
Loss for E

In [145]:
test_loader=DataLoader(test_dataset,batch_size=16,shuffle=False)
correct=0
total=0

model.eval()
with torch.no_grad():
    for x_test,y_test in test_loader:
        outputs=model(x_test)
        _, predicted = torch.max(outputs,1)
        total += y_test.size(0)
        correct += (predicted == y_test).sum().item()

accuracy = 100 * correct / total

print(f"Accuracy: {accuracy:.2f}%")

Accuracy: 100.00%
